In [3]:
from pathlib import Path

import pandas as pd


INPUT_PATH = Path("../data/master/modelling_features.csv")
OUTPUT_PATH = Path("../data/eda/sustainable_club_examples.csv")

BIG_SIX = {
    "Arsenal",
    "Chelsea",
    "Liverpool",
    "Manchester City",
    "Manchester United",
    "Tottenham",
}


df = pd.read_csv(INPUT_PATH)

# Exclude Big Six for Sunderland-relevant peer group
df = df[~df["team_name"].isin(BIG_SIX)].copy()

# Remove rows where final position is unavailable
df = df.dropna(subset=["team_name", "year", "position"])

# Analyse only clubs with at least 3 PL seasons in the sample
club_summary = (
    df.groupby("team_name")
    .agg(
        seasons_in_sample=("year", "nunique"),
        avg_finish=("position", "mean"),
        median_finish=("position", "median"),
        best_finish=("position", "min"),
        worst_finish=("position", "max"),
        finishes_8_to_15=("position", lambda x: x.between(8, 15).sum()),
        pct_finishes_8_to_15=("position", lambda x: x.between(8, 15).mean()),
        relegation_zone_finishes=("position", lambda x: (x >= 18).sum()),
        at_risk_finishes=("position", lambda x: (x >= 16).sum()),
        avg_npxGD=("npxGD", "mean"),
        avg_xGA=("xGA", "mean"),
        avg_xpts=("xpts", "mean"),
        avg_wage_rank=("wage_rank", "mean"),
        avg_revenue_rank=("revenue_rank", "mean"),
        avg_wage_to_revenue=("wage_to_revenue", "mean"),
        avg_net_spend=("net_spend", "mean"),
        avg_age=("avg_age", "mean"),
        manager_change_rate=("manager_change_flag", "mean"),
    )
    .reset_index()
)

sustainable_examples = club_summary[
    (club_summary["seasons_in_sample"] >= 3)
    & (club_summary["avg_finish"].between(8, 15))
    & (club_summary["pct_finishes_8_to_15"] >= 0.5)
    & (club_summary["relegation_zone_finishes"] == 0)
].copy()

sustainable_examples = sustainable_examples.sort_values(
    ["pct_finishes_8_to_15", "avg_finish"],
    ascending=[False, True],
)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
sustainable_examples.to_csv(OUTPUT_PATH, index=False)

print("Sustainable club examples:")
print(
    sustainable_examples[
        [
            "team_name",
            "seasons_in_sample",
            "avg_finish",
            "median_finish",
            "best_finish",
            "worst_finish",
            "pct_finishes_8_to_15",
            "avg_npxGD",
            "avg_wage_rank",
            "avg_age",
            "manager_change_rate",
        ]
    ].to_string(index=False)
)

print(f"\nSaved to {OUTPUT_PATH}")

Sustainable club examples:
              team_name  seasons_in_sample  avg_finish  median_finish  best_finish  worst_finish  pct_finishes_8_to_15  avg_npxGD  avg_wage_rank   avg_age  manager_change_rate
         Crystal Palace                  9   12.222222           12.0           10            14              1.000000  -8.052571      12.000000 26.100202             0.444444
              Brentford                  4   12.000000           11.5            9            16              0.750000   2.888489      18.750000 24.157376             0.000000
               West Ham                  9   11.111111           11.0            6            16              0.666667 -10.516119      11.000000 26.202010             0.333333
                Everton                  9   11.222222           11.0            7            17              0.666667  -4.460225       9.444444 25.541556             0.555556
       Newcastle United                  8    9.875000           11.5            4           